# Chapter 18 — GPU Kernel Construction

Part 7: Kernel construction and optimization.

**Ladder:** host-side mapping simulation (Method 01) → one Triton kernel per mapping — vector, row, column, tile, 3-D grid (Method 02) → persistent grid-stride and two-launch pipelines (Method 03).

**Thread.**

- **Recurring idea.** A kernel is a work-assignment map: the grid is the shape of the ownership relation, and every program owns a disjoint rectangle of the output.
- **Chain.** Vector → row → column → tile → 1-D/2-D/3-D grids → grid-stride → boundaries → multiple launches.
- **Check.** Every mapping is backed by an `assert`, and every tail is masked on both the load and the store.

**Study tip.**

- Fix $A\in\mathbb R^{5\times7}$ with $A_{ij}=7i+j+1$ and $v\in\mathbb R^{3\times4}$ with $v_{ij}=4i+j+1$:

$$
A=\begin{bmatrix}
1&2&3&4&5&6&7\\
8&9&10&11&12&13&14\\
15&16&17&18&19&20&21\\
22&23&24&25&26&27&28\\
29&30&31&32&33&34&35
\end{bmatrix}
$$

- Write four maps by hand: per row (5 programs), per $3\times4$ tile (grid $2\times2$), per element (35 programs), and a two-program grid-stride loop.
- List the program ids, owned rows and columns, and masked lanes; every hand value is checked later by an `assert`.

**Learning objectives**

- Choose a work assignment — vector, row, column, tile, or persistent worker — before writing the kernel body.
- Derive the launch grid for 1-D, 2-D, and 3-D mappings, including the **launch-grid rule** $P=\lceil N/B\rceil$ on every axis.
- Write the program-to-element map for a $5\times7$ matrix and a $3\times4$ vector by hand.
- Mask every non-divisible axis on both the load and the store, and choose `other` from the next reduction.
- Convert a fixed grid into a grid-stride loop and explain when that beats one program per tile.
- Decide between two launches and one fused kernel from traffic and launch overhead.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../ch17_control_flow/17_complete.ipynb) · [Next →](../ch19_tiling/19_complete.ipynb)

**Estimated time:** 80–100 min · **Difficulty:** Intermediate–Advanced

**Prerequisites:** Chapter 2 (program ids and launch grids), Chapter 6 (2-D offsets), Chapter 7 (masked loads and `other`).

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Concept diagrams

- **Four work assignments.** The grid is the shape of the ownership relation, not the shape of the data.

```text
per vector (flat)          per row                 per column              per tile
+--+--+--+--+              +----------------+      +--+--+--+--+          +-----+-----+
|p0|p0|p0|p0|              | row 0 = pid 0  |      |c0|c1|c2|c3|          |p00  |p01  |
+--+--+--+--+              +----------------+      |  |  |  |  |          |     |     |
|p1|p1|p1|p1|              | row 1 = pid 1  |      |  |  |  |  |          +-----+-----+
+--+--+--+--+              +----------------+      |  |  |  |  |          |p10  |p11  |
|p2|p2|..|..|              | ...            |      |  |  |  |  |          |     |     |
+--+--+--+--+              +----------------+      +--+--+--+--+          +-----+-----+
```

- **Launch grids.** A launch is a tuple of axis lengths; `tl.program_id(k)` reads one coordinate.

```text
1-D:  grid = (P,)              pid = program_id(0)           -> flat tile
2-D:  grid = (P_m, P_n)        (pm, pn) = (id(0), id(1))     -> row/col tile
3-D:  grid = (B, P_m, P_n)     (b, pm, pn) = (id(0), id(1), id(2))
```

- **Grid-stride.** With $P$ programs and block $B$, program $p$ visits starts $pB, pB+PB, pB+2PB,\ldots$ until $N$.

```text
p=0: [0..B-1]   [PB..PB+B-1]   [2PB..2PB+B-1] ...
p=1: [B..2B-1]  [PB+B..PB+2B-1] ...
     \___ one wave ___/  \___ next wave ___/
```

- **Bridge.** The loop converts a fixed launch into as many tiles as the data needs.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import pytest
import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, cdiv, require_gpu, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)

# Part I — Mathematics

- **Connection.** Builds on Chapter 2 (program ids and launch tuples) and Chapter 6 (2-D offsets and row-major addresses).
- **Focus.** Every section answers one question — which program owns which element — and derives the grid, mask, and address from that answer.
- **Order.** 18.1 vector → 18.2 row → 18.3 column → 18.4 tile → 18.5–18.7 grid rank → 18.8 grid-stride → 18.9 boundaries → 18.10 multiple launches.

## 18.1 One program per vector

- **Connection.** Chapter 2's flat 1-D grid, now written as an explicit ownership map.
- **Rule.** Program $p$ owns the contiguous block $V_p=\{pB+j:0\le j\lt B\}\cap[0,N)$; element $i$ has owner $p=\lfloor i/B\rfloor$ and lane $j=i\bmod B$.
- **Formula.** $P=\lceil N/B\rceil$.
- **Example.** $N=10$, $B=4$ gives $P=3$.

| pid | offsets | mask | live |
|---:|---|---|---|
| 0 | 0,1,2,3 | T,T,T,T | 0,1,2,3 |
| 1 | 4,5,6,7 | T,T,T,T | 4,5,6,7 |
| 2 | 8,9,10,11 | T,T,F,F | 8,9 |

- **Where.** Elementwise workhorse: vector add, activations, scaling; writes are disjoint and adjacent lanes coalesce.
- **Triton.** `offs = pid * BLOCK + tl.arange(0, BLOCK)`; `mask = offs < N`; guard both `tl.load` and `tl.store`.
- **Failure.** `grid=(N // B,)` drops the tail when $B\nmid N$; use `triton.cdiv(N, B)`.
- **Bridge.** A row keeps the same block but fixes one matrix row and varies only the column.

## 18.2 One program per row

- **Connection.** 18.1's vector block, with the block fixed to one matrix row instead of a flat range.
- **Rule.** Program $p$ owns row $p$: $R_p=\{(p,j):0\le j\lt N\}$. With column block $B_N$ it covers $j\in[p_nB_N,(p_n+1)B_N)$ and masks $j\lt N$.
- **Formula.** $P=M$.
- **Example.** $A$ ($5\times7$), $B_N=8$: 5 programs. Program 2 owns $A[2,:]=(15,\dots,21)$; lane 7 of its 8-lane tile is masked.
- **Where.** Row-wise normalization, row scaling $y_i=s_i x_i$, softmax statistics, layer norm; the whole row stays in one program, so no cross-program reduction is needed.
- **Triton.** `row = tl.program_id(0)`, `cols = tl.arange(0, BLOCK_N)`, `ptr = row * N + cols`, `mask = (row < M) & (cols < N)`.
- **Failure.** `BLOCK_N < N` without a column loop writes only the first `BLOCK_N` columns; use `BLOCK_N >= N` or a second grid axis / inner loop.
- **Bridge.** A column map swaps the two axes and exposes the coalescing cost of a row-major layout.

## 18.3 One program per column

- **Connection.** 18.2 transposed: program $p$ now owns a fixed column and walks the rows.
- **Rule.** $C_p=\{(i,p):0\le i\lt M\}$. In row-major storage the address is $iN+p$, so consecutive lanes of the program are $N$ elements apart.
- **Formula.** $P=N$.

$$
v=\begin{bmatrix}1&2&3&4\\5&6&7&8\\9&10&11&12\end{bmatrix}
$$

- **Example.** For $v$ ($3\times4$) there are 4 programs. Program 1 owns $(2,6,10)$; program 3 owns $(4,8,12)$; $B_M=4$ masks the fourth lane because $M=3$.
- **Where.** Column statistics, per-channel scale, and transposes. It is the right map for down-column reductions but the wrong map for bandwidth: a stride of $N$ touches one cache line per lane.
- **Triton.** `col = tl.program_id(0)`, `rows = tl.arange(0, BLOCK_M)`, `ptr = rows * N + col`, `mask = (rows < M) & (col < N)`.
- **Failure.** Swapping the two axes (`row = program_id(1)` with a 1-D grid) reads out of range or silently transposes the result; state the convention once and keep it.
- **Bridge.** A tile owns both a row block and a column block, recovering 2-D locality.

## 18.4 One program per tile

- **Connection.** 18.2 and 18.3 combined: a program now owns a rectangle, not a whole row or column.
- **Rule.** Program $(p_m,p_n)$ owns the tile $T_{p_m,p_n}=[p_mB_M,(p_m+1)B_M)\times[p_nB_N,(p_n+1)B_N)$.
- **Formula.** $P_m=\lceil M/B_M\rceil$, $P_n=\lceil N/B_N\rceil$.
- **Example.** $A$ ($5\times7$), $B_M=3$, $B_N=4$: grid $2\times2$.

| pid $(p_m,p_n)$ | rows | cols | row mask | col mask |
|---|---|---|---|---|
| (0,0) | 0,1,2 | 0,1,2,3 | T,T,T | T,T,T,T |
| (0,1) | 0,1,2 | 4,5,6,7 | T,T,T | T,T,T,F |
| (1,0) | 3,4,5 | 0,1,2,3 | T,T,F | T,T,T,T |
| (1,1) | 3,4,5 | 4,5,6,7 | T,T,F | T,T,T,F |

- **Totals.** 48 tile slots for 35 elements, so 13 lanes are masked.
- **Where.** Matrix multiply, convolution, attention, and tiled transpose; a tile gives 2-D locality, register reuse, and a path to tensor cores.
- **Triton.** `rows = pm*BM + tl.arange(0,BM)`, `cols = pn*BN + tl.arange(0,BN)`, `mask = (rows[:,None] < M) & (cols[None,:] < N)`, `ptr = rows[:,None]*N + cols[None,:]`.
- **Failure.** A wrong-rank mask — `(rows < M) & (cols < N)` broadcasts two `(B,)` vectors to `(B,B)` instead of `(B_M,B_N)` — or a transposed store offset; both fail loudly only when $B_M\ne B_N$.
- **Bridge.** Grid rank is separate from tile rank: one tile can be launched in 1-D, 2-D, or 3-D.

## 18.5 1-D launch grids

- **Connection.** 18.1's flat map is the canonical 1-D grid.
- **Rule.** A 1-D grid is one axis of $P$ programs; program $p$ owns $[pB,(p+1)B)$.
- **Formula.** $P=\lceil N/B\rceil$.
- **Example.** $N=35$, $B=8$: $P=5$. The last program owns offsets $32,\dots,39$ with mask `T,T,T,F,F,F,F,F`.
- **Where.** Elementwise kernels, vector add, and anything linearizable; it is the cheapest grid to reason about.
- **Triton.** `kernel[(triton.cdiv(n, BLOCK),)](...)`.
- **Failure.** `N // B` truncates; a grid larger than necessary is correct only if the mask guards every access, and it wastes scheduling slots.
- **Bridge.** A 2-D grid gives the tile its natural row and column coordinates.

## 18.6 2-D launch grids

- **Connection.** 18.4's tile map expressed as a 2-D launch tuple.
- **Rule.** One program per tile: $(p_m,p_n)\mapsto T_{p_m,p_n}$.
- **Formula.** $(P_m,P_n)=(\lceil M/B_M\rceil,\lceil N/B_N\rceil)$.
- **Example.** $A$ ($5\times7$) with $3\times4$ tiles gives $(2,2)$ and 4 programs; $v$ ($3\times4$) with $2\times2$ tiles also gives $(2,2)$, but program $(1,1)$ has a masked row.
- **Where.** 2-D stencils, tiled copy, transpose, matmul; each dimension carries its own mask.
- **Triton.** `pm, pn = tl.program_id(0), tl.program_id(1)`; launch `grid=(cdiv(M,BM), cdiv(N,BN))`.
- **Failure.** Exchanging `pm` and `pn` on a non-square grid covers the wrong rectangle; with $M\ne N$ it is an out-of-bounds bug, not merely a slow path.
- **Bridge.** A leading batch axis turns the same tile into a 3-D launch.

## 18.7 3-D launch grids

- **Connection.** 18.6's tile plus a leading batch coordinate.
- **Rule.** Grid $(B,P_m,P_n)$ maps $(b,p_m,p_n)$ to batch $b$ and tile $T_{p_m,p_n}$.
- **Formula.** Offset $bMN+iN+j$.
- **Example.** Two copies of $A$: $B=2$, $B_M=3$, $B_N=4$ gives $(2,2,2)=8$ programs. Program $(1,1,1)$ owns batch 1, rows $3,4,5$, cols $4,5,6,7$; row 5 and col 7 are masked.
- **Where.** Batched matmul, attention with `(batch, head, tile)`, and per-image convolution; the batch axis removes a Python loop.
- **Triton.** `b, pm, pn = tl.program_id(0), tl.program_id(1), tl.program_id(2)`.
- **Failure.** Forgetting the batch stride $MN$ makes every batch overwrite batch 0; reversing the axis order silently permutes the output.
- **Bridge.** A fixed grid can also walk work with a runtime loop instead of enumerating every tile.

## 18.8 Grid-stride processing

- **Connection.** 18.5's flat grid, now launched with fewer programs than tiles.
- **Rule.** Fix $P$ programs; program $p$ walks starts $pB+kPB$ until $N$.
- **Formula.** $W_p=\{pB+kPB+j:0\le j\lt B,\ k\ge0\}$; the loop is `range(pid*BLOCK, n, num_programs*BLOCK)`.
- **Example.** $N=35$, $B=4$, $P=2$: stride $PB=8$.

| pid | starts | tiles | masked flat index |
|---:|---|---|---|
| 0 | 0, 8, 16, 24, 32 | $[0,4),[8,12),[16,20),[24,28),[32,36)$ | 35 |
| 1 | 4, 12, 20, 28 | $[4,8),[12,16),[20,24),[28,32)$ | none |

- **Totals.** 9 tiles, exactly $\lceil35/4\rceil$, and one masked lane.
- **Where.** Persistent kernels, variable-length inputs, and occupancy control; a fixed grid removes wave quantization and lets a program reuse registers across tiles.
- **Triton.** `step = tl.num_programs(0) * BLOCK`; iterate `range(pid * BLOCK, n_elements, step)`; mask every iteration.
- **Failure.** `range(pid*BLOCK, n, num_programs)` re-reads or skips tiles; starting at `pid * P * BLOCK` leaves the first wave unwritten.
- **Bridge.** Every loop still needs the boundary rule, which is the next section.

## 18.9 Boundary handling

- **Connection.** Every previous mapping assumed divisible shapes; this section adds the missing mask.
- **Rule.** Give every non-divisible axis its own mask; the tile mask is the outer product $m_{i,j}=[i\lt M]\land[j\lt N]$.
- **Formula.** Guard both `tl.load` and `tl.store` with $m$.
- **Example.** Tile $(p_m,p_n)=(1,1)$ of $A$ has rows $3,4,5$ and cols $4,5,6,7$, so $m=[[1,1,1,0],[1,1,1,0],[0,0,0,0]]$.
- **Where.** Almost no real shape divides evenly; the mask is the correctness contract for the last tile of every mapping.
- **Triton.** `row_mask[:, None] & col_mask[None, :]`; choose `other` from the next reduction: `0.0` for sum/add, `-inf` for max, `+inf` for min.
- **Failure.** Masking only the store still performs an out-of-bounds load; applying a `(B_M,)` mask directly to a `(B_M,B_N)` pointer tile broadcasts the wrong way.
- **Bridge.** With the map and mask fixed, the remaining choice is how many launches to use.

## 18.10 Multiple kernel launches

- **Connection.** 18.8's pipeline can be one persistent kernel or several launches; this section prices the split.
- **Rule.** Two kernels $Y=f(X)$, $Z=g(Y)$ need two launches and an intermediate $Y$; on one stream the second launch sees the first kernel's writes.
- **Formula.** $Z=2X+1$ in two steps $Y=2X$, $Z=Y+1$.
- **Example.** For $A$ ($5\times7$) and `float32`, the intermediate costs $4\cdot35\cdot4=560$ bytes versus $2\cdot35\cdot4=280$ bytes for the fused form.
- **Where.** Required for a global barrier, a different grid, or a different layout (row scale then transpose); wasteful for same-shape elementwise stages.
- **Triton.** Call kernels in order on the same stream; there is no implicit device-wide sync inside one kernel. Use separate streams only with an explicit event.
- **Failure.** Many tiny kernels make launch overhead dominate; assuming a second kernel can read a first kernel's output without stream ordering is a race.
- **Bridge.** Fusion is the default for elementwise chains; the launch split is reserved for a barrier or a layout change.

**Where this leads.** Part II simulates these maps on the host, Part III transcribes them into Triton, and Part IV makes the grid persistent and pipelines two launches.

In [ ]:
M, N, BM, BN, B = 5, 7, 3, 4, 2

assert cdiv(M, BM) == 2 and cdiv(N, BN) == 2
assert cdiv(M, BM) * cdiv(N, BN) == 4
assert M * N == 35
assert cdiv(N, 8) == 1                      # one program covers a 7-wide row
assert cdiv(35, 8) == 5                     # 1-D grid example in 18.5
assert (torch.arange(32, 40) < 35).tolist() == [True, True, True] + [False] * 5
assert cdiv(3, 4) == 1                      # one column program covers M=3
assert cdiv(35, 4) == 9                     # grid-stride tiles for B=4
assert B * cdiv(M, BM) * cdiv(N, BN) == 8    # 3-D grid
assert cdiv(3, 2) * cdiv(4, 2) == 4          # 2x2 tiles for the 3x4 vector
assert cdiv(8, 4) == 2 and cdiv(9, 4) == 3   # ceil, not floor
assert cdiv(0, 4) == 0

## Chapter map

- **Reading.** Each row is one ownership map; the last column names the notebook kernel or demo.

| § | Mapping | Grid formula | Boundary rule | Kernel/demo |
|---|---|---|---|---|
| **18.1** | one program per vector | $P=\lceil N/B\rceil$ | `offs < N` | worked-example assert; grid-stride baseline |
| **18.2** | one program per row | $P=M$ | `cols < N` | `row_scale_kernel` |
| **18.3** | one program per column | $P=N$ | `rows < M` | `column_scale_kernel` |
| **18.4** | one program per tile | $(\lceil M/B_M\rceil,\lceil N/B_N\rceil)$ | row mask $\land$ col mask | `tile_copy_kernel` |
| **18.5** | 1-D launch grid | $(P,)$ | one axis | grid-stride baseline |
| **18.6** | 2-D launch grid | $(P_m,P_n)$ | two axes | `tile_copy_kernel` |
| **18.7** | 3-D launch grid | $(B,P_m,P_n)$ | batch + two axes | `batch_copy_kernel` |
| **18.8** | grid-stride | $P$ fixed, step $PB$ | per-iteration mask | `grid_stride_kernel` |
| **18.9** | boundary handling | any of the above | all axes | every kernel |
| **18.10** | multiple launches | two grids in sequence | each stage own rule | `two_launch_scale_transpose` |

## Worked numerical example — a $5\times7$ matrix and a $3\times4$ vector

- **Setup.**

$$
A=\begin{bmatrix}
1&2&3&4&5&6&7\\
8&9&10&11&12&13&14\\
15&16&17&18&19&20&21\\
22&23&24&25&26&27&28\\
29&30&31&32&33&34&35
\end{bmatrix},
\qquad
v=\begin{bmatrix}1&2&3&4\\5&6&7&8\\9&10&11&12\end{bmatrix}.
$$

- **Per row.** $P=M=5$; program $p$ owns all seven entries of row $p$, so $p\mapsto\{(p,j):0\le j\lt 7\}$. With $B_N=8$ every program has one masked lane ($j=7$).
- **Per $3\times4$ tile.** Grid $(\lceil5/3\rceil,\lceil7/4\rceil)=(2,2)$.

| $(p_m,p_n)$ | flat pid | rows | cols | masked cells |
|---|---|---:|---|---:|
| (0,0) | 0 | 0,1,2 | 0,1,2,3 | 0 |
| (0,1) | 1 | 0,1,2 | 4,5,6,7 | 3 |
| (1,0) | 2 | 3,4,5 | 0,1,2,3 | 4 |
| (1,1) | 3 | 3,4,5 | 4,5,6,7 | 6 |

- **Tile totals.** $0+3+4+6=13=48-35$; program $(1,1)$ masks row 5 and column 7.
- **Per element.** $P=MN=35$; program $q$ owns $(\lfloor q/7\rfloor,q\bmod7)$; $q=0\mapsto(0,0)$, $q=6\mapsto(0,6)$, $q=7\mapsto(1,0)$, $q=34\mapsto(4,6)$. No mask is needed, but 35 programs handle 35 values.
- **Grid-stride with two programs.** Flatten $A$ to $N=35$; $B=4$, $P=2$, stride $PB=8$.

| pid | starts | row/col of each start | tail mask |
|---:|---|---|---|
| 0 | 0,8,16,24,32 | $(0,0),(1,1),(2,2),(3,3),(4,4)$ | last tile `T,T,T,F` |
| 1 | 4,12,20,28 | $(0,4),(1,5),(2,6),(4,0)$ | none |

- **Tail.** Program 0's last tile covers flat $32,33,34,35$; flat 35 would be $(5,0)$ and is masked, which is why the loop still carries `offs < N`.
- **The $3\times4$ vector with $2\times2$ tiles.** Rows $0,1$ are full and rows $2,3$ carry the mask ($16-12=4$ slots); per column, program 3 owns $(4,8,12)$.

In [ ]:
A = torch.arange(1, 36, dtype=torch.float32).view(5, 7)
v = torch.arange(1, 13, dtype=torch.float32).view(3, 4)

assert A.shape == (5, 7) and v.shape == (3, 4)
assert A[0].tolist() == [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
assert A[4].tolist() == [29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0]

# One program per row: five programs, seven live columns each.
assert [p for p in range(A.shape[0])] == [0, 1, 2, 3, 4]
assert [A[p, 2].item() for p in range(5)] == [3.0, 10.0, 17.0, 24.0, 31.0]

# One program per 3x4 tile.
tiles = {
    (pm, pn): (
        [pm * 3 + i for i in range(3)],
        [pn * 4 + j for j in range(4)],
    )
    for pm in range(cdiv(5, 3))
    for pn in range(cdiv(7, 4))
}
assert list(tiles) == [(0, 0), (0, 1), (1, 0), (1, 1)]
assert tiles[(0, 1)] == ([0, 1, 2], [4, 5, 6, 7])
assert tiles[(1, 1)] == ([3, 4, 5], [4, 5, 6, 7])


def masked(rows, cols, m=5, n=7):
    return sum(1 for i in rows for j in cols if i >= m or j >= n)


assert [masked(*tiles[k]) for k in tiles] == [0, 3, 4, 6]
assert sum(masked(*tiles[k]) for k in tiles) == 13
assert sum(len(r) * len(c) for r, c in tiles.values()) == 48

# One program per element.
assert [q // 7 for q in range(35)][-1] == 4
assert [(q // 7, q % 7) for q in range(35)][7] == (1, 0)
assert [(q // 7, q % 7) for q in range(35)][34] == (4, 6)
assert set((q // 7, q % 7) for q in range(35)) == {
    (i, j) for i in range(5) for j in range(7)
}

# Two-program grid-stride over the flattened matrix.
starts = {p: list(range(p * 4, 35, 2 * 4)) for p in range(2)}
assert starts[0] == [0, 8, 16, 24, 32]
assert starts[1] == [4, 12, 20, 28]
assert sum(len(s) for s in starts.values()) == 9
assert (torch.arange(32, 36) < 35).tolist() == [True, True, True, False]
assert [(q // 7, q % 7) for q in range(32, 36)] == [(4, 4), (4, 5), (4, 6), (5, 0)]

# The 3x4 vector with 2x2 tiles.
vtiles = [
    ([pm * 2 + i for i in range(2)], [pn * 2 + j for j in range(2)])
    for pm in range(cdiv(3, 2))
    for pn in range(cdiv(4, 2))
]
assert sum(masked(r, c, m=3, n=4) for r, c in vtiles) == 4
assert [v[i, 3].item() for i in range(3)] == [4.0, 8.0, 12.0]

# Part II — Method 01: host-side mapping simulation

- **Connection.** Part I fixed the ownership maps; Method 01 executes the same relation one program at a time on the CPU.
- **Rule.** No launch: each map yields the full tile including masked lanes, and the host skips a masked lane with `if`.
- **Maps.** `map_vector` (flat `pB+j`), `map_rows` (row `p`), `map_cols` (column `p`), `map_tiles` (`(p_m,p_n)`), `map_grid_stride` (starts `pB+kPB`), `map_batch` (`(b,p_m,p_n)`).
- **Where.** This is the correctness oracle: if a map is wrong, the Python simulation and the Triton kernel disagree on the same input.
- **Difference from Triton.** The host loop removes a masked lane; the kernel keeps the lane and masks the memory access.
- **Bridge.** Method 02 transcribes the same maps into kernels.

In [ ]:
def map_vector(n, block):
    return [(p, list(range(p * block, min((p + 1) * block, n))))
            for p in range(cdiv(n, block))]


def map_rows(m):
    return [(p, [p]) for p in range(m)]


def map_cols(n):
    return [(p, [p]) for p in range(n)]


def map_tiles(m, n, bm, bn):
    return [
        ((pm, pn),
         [pm * bm + i for i in range(bm)],
         [pn * bn + j for j in range(bn)])
        for pm in range(cdiv(m, bm))
        for pn in range(cdiv(n, bn))
    ]


def map_grid_stride(n, block, programs):
    return [(p, list(range(p * block, n, programs * block)))
            for p in range(programs)]


def map_batch(b, m, n, bm, bn):
    return [
        (bi, (pm, pn), [pm * bm + i for i in range(bm)],
         [pn * bn + j for j in range(bn)])
        for bi in range(b)
        for pm in range(cdiv(m, bm))
        for pn in range(cdiv(n, bn))
    ]


assert map_vector(10, 4) == [(0, [0, 1, 2, 3]), (1, [4, 5, 6, 7]), (2, [8, 9])]
assert map_rows(5) == [(p, [p]) for p in range(5)]
assert map_cols(4) == [(p, [p]) for p in range(4)]
assert map_tiles(5, 7, 3, 4)[1] == ((0, 1), [0, 1, 2], [4, 5, 6, 7])
assert map_grid_stride(35, 4, 2)[0] == (0, [0, 8, 16, 24, 32])
assert len(map_batch(2, 5, 7, 3, 4)) == 8

In [ ]:
def host_vector_scale(x, factor, block):
    out = torch.zeros_like(x)
    src, dst = x.reshape(-1), out.reshape(-1)
    for _, live in map_vector(x.numel(), block):
        for i in live:
            dst[i] = src[i] * factor
    return out


def host_row_scale(x, s, block_n):
    m, n = x.shape
    out = torch.zeros_like(x)
    for p, _ in map_rows(m):
        for j in range(block_n):
            if j < n:
                out[p, j] = x[p, j] * s[p]
    return out


def host_col_scale(x, factor, block_m):
    m, n = x.shape
    out = torch.zeros_like(x)
    for p, _ in map_cols(n):
        for i in range(block_m):
            if i < m:
                out[i, p] = x[i, p] * factor
    return out


def host_tile_copy(x, bm, bn, transpose=False):
    m, n = x.shape
    out = torch.zeros((n, m) if transpose else (m, n), dtype=x.dtype)
    for _, rows, cols in map_tiles(m, n, bm, bn):
        for i in rows:
            for j in cols:
                if i < m and j < n:
                    if transpose:
                        out[j, i] = x[i, j]
                    else:
                        out[i, j] = x[i, j]
    return out


def host_grid_stride(x, shift, block, programs):
    out = torch.zeros_like(x)
    src, dst = x.reshape(-1), out.reshape(-1)
    for _, starts in map_grid_stride(x.numel(), block, programs):
        for start in starts:
            for j in range(block):
                i = start + j
                if i < x.numel():
                    dst[i] = src[i] + shift
    return out


def host_batch_scale(x, factor, bm, bn):
    b, m, n = x.shape
    out = torch.zeros_like(x)
    for bi, _, rows, cols in map_batch(b, m, n, bm, bn):
        for i in rows:
            for j in cols:
                if i < m and j < n:
                    out[bi, i, j] = x[bi, i, j] * factor
    return out


def test_method01():
    A = torch.arange(1, 36, dtype=torch.float32).view(5, 7)
    v = torch.arange(1, 13, dtype=torch.float32).view(3, 4)
    s = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
    xb = torch.arange(2 * 5 * 7, dtype=torch.float32).view(2, 5, 7)

    assert torch.equal(host_vector_scale(A, 2.0, 8), 2.0 * A)
    assert torch.equal(host_row_scale(A, s, 8), A * s[:, None])
    assert torch.equal(host_col_scale(v, 2.0, 4), 2.0 * v)
    assert torch.equal(host_tile_copy(A, 3, 4), A)
    assert torch.equal(host_tile_copy(A, 3, 4, transpose=True), A.T)
    assert torch.equal(host_grid_stride(A, 5.0, 4, 2), A + 5.0)
    assert torch.equal(host_batch_scale(xb, 3.0, 3, 4), 3.0 * xb)

    # Each element is written exactly once by every non-overlapping map.
    counts = torch.zeros(35, dtype=torch.int32)
    for _, live in map_vector(35, 4):
        for i in live:
            counts[i] += 1
    assert counts.tolist() == [1] * 35

# Part III — Method 02: Triton kernels for each mapping

- **Connection.** Method 01's host maps become device kernels.
- **Rule.** Every kernel decides the program coordinate, forms offsets, forms the mask, loads, computes, and stores.
- **Formula.** $P_k=\lceil \text{size}_k/B_k\rceil$ for every grid axis $k$.
- **Scope.** Same $5\times7$ and $3\times4$ shapes as the worked example, plus odd shapes to exercise boundaries.
- **Implementation note.** `tl.arange` needs power-of-two blocks, so a $3\times4$ logical tile runs on a $4\times4$ block with a lane mask.
- **Bridge.** Row scale is the first transcription.

## 18.2 in Triton — row scale

- **Connection.** Part I's 18.2 row map.
- **Rule.** $Y_{ij}=s_iX_{ij}$; one program per row.
- **Example.** $A$ ($5\times7$): 5 programs of 8 lanes, eighth lane masked.
- **Where.** The scalar $s_p$ is loaded once per program and broadcasts over the row.
- **Triton.** `row = program_id(0)`; `cols = arange(0, BLOCK_N)`; mask `(row<M)&(cols<N)`.
- **Failure.** A too-small `BLOCK_N` without a loop drops columns.
- **Bridge.** The column kernel swaps the axes.

In [ ]:
@triton.jit
def row_scale_kernel(x_ptr, s_ptr, out_ptr, M, N, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    mask = (row < M) & (cols < N)
    x = tl.load(x_ptr + row * N + cols, mask=mask, other=0.0)
    s = tl.load(s_ptr + row)
    tl.store(out_ptr + row * N + cols, x * s, mask=mask)


def row_scale(x, s, BLOCK_N=None):
    x = x.contiguous()
    s = s.contiguous()
    m, n = x.shape
    assert s.shape == (m,), "one scale per row"
    block_n = BLOCK_N or triton.next_power_of_2(n)
    out = torch.empty_like(x)
    if m == 0 or n == 0:
        return out
    row_scale_kernel[(m,)](x, s, out, m, n, BLOCK_N=block_n)
    return out

In [ ]:
def test_row_scale():
    require_gpu("row scale")
    x = torch.randn(5, 7, device=DEVICE)
    s = torch.randn(5, device=DEVICE)
    torch.testing.assert_close(row_scale(x, s), x * s[:, None])

    x = torch.randn(19, 37, device=DEVICE)
    s = torch.randn(19, device=DEVICE)
    torch.testing.assert_close(row_scale(x, s, BLOCK_N=64), x * s[:, None])

## 18.3 in Triton — column scale

- **Connection.** Part I's 18.3 column map.
- **Rule.** Program $p$ walks rows of column $p$ with stride $N$.
- **Example.** $v$ ($3\times4$): 4 programs and $B_M=4$ lanes, so the last lane of every program is masked.
- **Where.** Natural for column statistics even though the loads are strided.
- **Triton.** `col = program_id(0)`; `rows = arange(0, BLOCK_M)`; mask `(rows<M)&(col<N)`.
- **Failure.** Axis swap or a stride mistake reads the wrong column.
- **Bridge.** The tile kernel recovers contiguous access inside each tile.

In [ ]:
@triton.jit
def column_scale_kernel(x_ptr, out_ptr, M, N, BLOCK_M: tl.constexpr):
    col = tl.program_id(0)
    rows = tl.arange(0, BLOCK_M)
    mask = (rows < M) & (col < N)
    x = tl.load(x_ptr + rows * N + col, mask=mask, other=0.0)
    tl.store(out_ptr + rows * N + col, 2.0 * x, mask=mask)


def column_scale(x, BLOCK_M=None):
    m, n = x.shape
    block_m = BLOCK_M or triton.next_power_of_2(m)
    out = torch.empty_like(x)
    if m == 0 or n == 0:
        return out
    column_scale_kernel[(n,)](x, out, m, n, BLOCK_M=block_m)
    return out


def test_column_scale():
    require_gpu("column scale")
    v = torch.arange(1, 13, dtype=torch.float32, device=DEVICE).view(3, 4)
    torch.testing.assert_close(column_scale(v), 2.0 * v)

    x = torch.randn(17, 29, device=DEVICE)
    torch.testing.assert_close(column_scale(x, BLOCK_M=32), 2.0 * x)

## 18.4 in Triton — tiled copy and transpose

- **Connection.** Part I's 18.4 tile map, with 18.6's 2-D grid.
- **Rule.** One kernel serves copy and transpose via `TRANSPOSE: tl.constexpr`.
- **Formula.** Load $iN+j$; transposed store $jM+i$.
- **Where.** The mask is the same for both because the logical tile does not change.
- **Triton.** `pm,pn = program_id(0), program_id(1)`; `rows = pm*TILE_M + arange(0,BLOCK_M)`; combine `(lm<TILE_M)&(rows<M)` with the column mask.
- **Failure.** `jN+i` writes into the wrong rectangle; a missing lane mask duplicates the pad row.
- **Bridge.** A batch axis turns the tile into a 3-D grid.

In [ ]:
@triton.jit
def tile_copy_kernel(x_ptr, out_ptr, M, N,
                     TILE_M: tl.constexpr, TILE_N: tl.constexpr,
                     BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
                     TRANSPOSE: tl.constexpr):
    pm = tl.program_id(0)
    pn = tl.program_id(1)
    lm = tl.arange(0, BLOCK_M)
    ln = tl.arange(0, BLOCK_N)
    rows = pm * TILE_M + lm
    cols = pn * TILE_N + ln
    row_mask = (lm < TILE_M) & (rows < M)
    col_mask = (ln < TILE_N) & (cols < N)
    mask = row_mask[:, None] & col_mask[None, :]
    ptrs = rows[:, None] * N + cols[None, :]
    x = tl.load(x_ptr + ptrs, mask=mask, other=0.0)
    if TRANSPOSE:
        tl.store(out_ptr + cols[None, :] * M + rows[:, None], x, mask=mask)
    else:
        tl.store(out_ptr + ptrs, x, mask=mask)


def tile_copy(x, tile_m=3, tile_n=4, transpose=False):
    m, n = x.shape
    out = torch.empty((n, m) if transpose else (m, n),
                      device=x.device, dtype=x.dtype)
    if m == 0 or n == 0:
        return out
    grid = (cdiv(m, tile_m), cdiv(n, tile_n))
    tile_copy_kernel[grid](
        x, out, m, n,
        TILE_M=tile_m, TILE_N=tile_n,
        BLOCK_M=triton.next_power_of_2(tile_m),
        BLOCK_N=triton.next_power_of_2(tile_n),
        TRANSPOSE=transpose,
    )
    return out

In [ ]:
def test_tile_copy():
    require_gpu("tile copy")
    x = torch.randn(5, 7, device=DEVICE)
    torch.testing.assert_close(tile_copy(x, 3, 4), x)
    torch.testing.assert_close(tile_copy(x, 3, 4, transpose=True), x.T)

    y = torch.randn(17, 29, device=DEVICE)
    torch.testing.assert_close(tile_copy(y, 8, 16), y)
    torch.testing.assert_close(tile_copy(y, 8, 16, transpose=True), y.T)

## 18.7 in Triton — batch, row, column

- **Connection.** Part I's 18.7 3-D grid.
- **Rule.** The batch axis is program id 0 and contributes the stride $MN$; the other two axes are the tile.
- **Formula.** Grid $(B,\lceil M/B_M\rceil,\lceil N/B_N\rceil)$.
- **Where.** The kernel copies each element, so a missing or duplicated write is visible in the output.
- **Triton.** `b,pm,pn = program_id(0), program_id(1), program_id(2)`; offset `b*M*N + rows[:,None]*N + cols[None,:]`.
- **Failure.** Forgetting $bMN$ overwrites batch 0.
- **Bridge.** Part IV keeps the grid fixed and loops over work.

In [ ]:
@triton.jit
def batch_copy_kernel(x_ptr, out_ptr, B, M, N,
                      TILE_M: tl.constexpr, TILE_N: tl.constexpr,
                      BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    b = tl.program_id(0)
    pm = tl.program_id(1)
    pn = tl.program_id(2)
    lm = tl.arange(0, BLOCK_M)
    ln = tl.arange(0, BLOCK_N)
    rows = pm * TILE_M + lm
    cols = pn * TILE_N + ln
    row_mask = (lm < TILE_M) & (rows < M)
    col_mask = (ln < TILE_N) & (cols < N)
    mask = (b < B) & row_mask[:, None] & col_mask[None, :]
    offs = b * M * N + rows[:, None] * N + cols[None, :]
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x, mask=mask)


def batch_copy(x, tile_m=3, tile_n=4):
    b, m, n = x.shape
    out = torch.empty_like(x)
    if b == 0 or m == 0 or n == 0:
        return out
    grid = (b, cdiv(m, tile_m), cdiv(n, tile_n))
    batch_copy_kernel[grid](
        x, out, b, m, n,
        TILE_M=tile_m, TILE_N=tile_n,
        BLOCK_M=triton.next_power_of_2(tile_m),
        BLOCK_N=triton.next_power_of_2(tile_n),
    )
    return out


def test_batch_copy():
    require_gpu("3-D grid")
    x = torch.randn(2, 5, 7, device=DEVICE)
    torch.testing.assert_close(batch_copy(x, 3, 4), x)

    y = torch.randn(3, 9, 17, device=DEVICE)
    torch.testing.assert_close(batch_copy(y, 4, 8), y)

# Part IV — Method 03: persistent and multi-launch pipelines

## 18.8 in Triton — grid-stride

- **Connection.** Part I's 18.8 grid-stride map, launched with a fixed number of programs.
- **Rule.** The loop bound is the data size, the step is the grid times the block, and the mask is recomputed each iteration.
- **Formula.** $pB+kPB$ for $k=0,1,\dots$ until $N$.
- **Example.** $N=35$, $B=4$, $P=2$: last start 32, one masked lane.
- **Tail-heavy tests.** $N=37$, $B=8$, $P=3$ leaves a mostly masked final tile; $N=3$, $P=8$ leaves most programs idle.
- **Where.** Useful when the tile count is unknown at launch, when the tail would create a partial wave, or when the kernel keeps state in registers across tiles.
- **Triton.** `step = tl.num_programs(0)*BLOCK`; `for start in range(pid*BLOCK, n, step)`.
- **Failure.** A step of `num_programs` instead of `num_programs*BLOCK` skips or re-reads tiles.
- **Bridge.** The last section splits the pipeline into two launches.

In [ ]:
@triton.jit
def grid_stride_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    step = tl.num_programs(0) * BLOCK
    for start in range(pid * BLOCK, n_elements, step):
        offs = start + tl.arange(0, BLOCK)
        mask = offs < n_elements
        x = tl.load(x_ptr + offs, mask=mask, other=0.0)
        tl.store(out_ptr + offs, x + 5.0, mask=mask)


def grid_stride_add(x, BLOCK=4, programs=2):
    out = torch.empty_like(x)
    if x.numel() == 0 or programs == 0:
        return out
    grid_stride_kernel[(programs,)](x, out, x.numel(), BLOCK=BLOCK)
    return out

In [ ]:
def test_grid_stride():
    require_gpu("grid stride")
    x = torch.arange(35, dtype=torch.float32, device=DEVICE)
    torch.testing.assert_close(grid_stride_add(x, BLOCK=4, programs=2), x + 5.0)

    x = torch.arange(37, dtype=torch.float32, device=DEVICE)
    torch.testing.assert_close(grid_stride_add(x, BLOCK=8, programs=3), x + 5.0)

    x = torch.randn(1000, device=DEVICE)
    torch.testing.assert_close(grid_stride_add(x, BLOCK=128, programs=4), x + 5.0)

    x = torch.tensor([1.0, 2.0, 3.0], device=DEVICE)
    torch.testing.assert_close(grid_stride_add(x, BLOCK=8, programs=8), x + 5.0)

## 18.10 in Triton — row scale then transpose

- **Connection.** 18.8's persistent loop plus 18.10's launch split.
- **Rule.** $Z=(s\odot X)^\top$ via $Y=s\odot X$ and $Z=Y^\top$.
- **Example.** Stage 1 launches $M=5$ row programs with $B_N=8$; stage 2 launches a $2\times2$ grid with $3\times4$ tiles.
- **Where.** The fused kernel removes $Y$: it loads $X$, multiplies by $s$ in registers, and stores transposed.
- **Formula.** $Q_{\text{two}}=4MNb$, $Q_{\text{fused}}=2MNb$; for $5\times7$, `float32`, 560 versus 280 bytes.
- **Triton.** Call `row_scale` then `tile_copy(..., transpose=True)`, or call the fused `scale_transpose_kernel`.
- **Failure.** Two launches pay intermediate traffic plus an extra launch latency; the fused form wins for same-shape elementwise stages.
- **Bridge.** The choosing-a-mapping section turns these costs into a decision table.

In [ ]:
@triton.jit
def scale_transpose_kernel(x_ptr, s_ptr, out_ptr, M, N,
                           TILE_M: tl.constexpr, TILE_N: tl.constexpr,
                           BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pm = tl.program_id(0)
    pn = tl.program_id(1)
    lm = tl.arange(0, BLOCK_M)
    ln = tl.arange(0, BLOCK_N)
    rows = pm * TILE_M + lm
    cols = pn * TILE_N + ln
    row_mask = (lm < TILE_M) & (rows < M)
    col_mask = (ln < TILE_N) & (cols < N)
    mask = row_mask[:, None] & col_mask[None, :]
    x = tl.load(x_ptr + rows[:, None] * N + cols[None, :], mask=mask, other=0.0)
    s = tl.load(s_ptr + rows, mask=row_mask, other=0.0)
    y = x * s[:, None]
    tl.store(out_ptr + cols[None, :] * M + rows[:, None], y, mask=mask)


def two_launch_scale_transpose(x, s, tile_m=3, tile_n=4):
    y = row_scale(x, s)
    return tile_copy(y, tile_m, tile_n, transpose=True)


def scale_transpose_fused(x, s, tile_m=3, tile_n=4):
    m, n = x.shape
    out = torch.empty((n, m), device=x.device, dtype=x.dtype)
    if m == 0 or n == 0:
        return out
    grid = (cdiv(m, tile_m), cdiv(n, tile_n))
    scale_transpose_kernel[grid](
        x, s, out, m, n,
        TILE_M=tile_m, TILE_N=tile_n,
        BLOCK_M=triton.next_power_of_2(tile_m),
        BLOCK_N=triton.next_power_of_2(tile_n),
    )
    return out

In [ ]:
def test_multiple_launches():
    require_gpu("two launches")
    x = torch.randn(5, 7, device=DEVICE)
    s = torch.randn(5, device=DEVICE)
    ref = (x * s[:, None]).T
    torch.testing.assert_close(two_launch_scale_transpose(x, s, 3, 4), ref)
    torch.testing.assert_close(scale_transpose_fused(x, s, 3, 4), ref)

    y = torch.randn(17, 29, device=DEVICE)
    t = torch.randn(17, device=DEVICE)
    ref = (y * t[:, None]).T
    torch.testing.assert_close(two_launch_scale_transpose(y, t, 8, 16), ref)
    torch.testing.assert_close(scale_transpose_fused(y, t, 8, 16), ref)

    # Traffic: two launches write and re-read the intermediate; the fused form does not.
    m, n, b = 5, 7, 4
    assert 4 * m * n * b == 560
    assert 2 * m * n * b == 280

## Choosing a mapping

- **Connection.** The mapping determines grid size, tail waste, coalescing, and launch count.
- **Forces.** Occupancy (enough programs), tail waste (masked lanes), coalescing (adjacent lanes touch adjacent addresses), launch count (kernels and intermediate traffic).
- **Formula.** Waste $=(P_mP_nB_MB_N-MN)/(P_mP_nB_MB_N)$.
- **Trade-off table.**

| Mapping | Grid | Coalescing on row-major | Tail waste | Strength |
|---|---|---|---|---|
| per vector | $\lceil N/B\rceil$ | perfect | $O(B)$ | elementwise, simplest |
| per row | $M$ | perfect along the row | $O(MB_N)$ if $B_N\gt N$ | row reductions, row scale |
| per column | $N$ | poor (stride $N$) | $O(NB_M)$ | column reductions, transpose |
| per tile | $P_mP_n$ | perfect inside a tile row | $P_mP_nB_MB_N-MN$ | 2-D reuse, matmul |
| 3-D | $BP_mP_n$ | same as tile per batch | same as tile | batched work, no Python loop |
| grid-stride | $P$ fixed | same as the underlying tile | at most one tile per program | persistence, unknown $N$ |

- **Rule of thumb.** Start with one program per tile, or per row for row-wise reductions. Move to grid-stride when the grid would be much larger than the number of resident programs, or when the problem size is dynamic. Fuse elementwise stages; keep two launches when a global barrier or a layout change is required.
- **Bridge.** The pitfalls and exercises apply the table.

In [ ]:
M, N, BM, BN, B = 5, 7, 3, 4, 2

rows_programs = M
tile_programs = cdiv(M, BM) * cdiv(N, BN)
element_programs = M * N
stride_programs = 2

assert (rows_programs, tile_programs, element_programs, stride_programs) == (5, 4, 35, 2)
assert rows_programs * 8 - M * N == 5          # row-block waste with BN=8
assert tile_programs * BM * BN - M * N == 13   # 3x4 tile waste
assert element_programs * 1 - M * N == 0        # one element per program
assert cdiv(35, 4) * 4 - M * N == 1             # one masked lane in grid-stride
assert B * tile_programs * BM * BN - B * M * N == 26

# Coalescing: a row is contiguous, a column is not.
A = torch.arange(1, 36, dtype=torch.float32).view(5, 7)
assert A[2].stride() == (1,)
assert A[:, 2].stride() == (7,)

## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| grid too small (`N // B`) | the tail is never written | `P = cdiv(N, B)` on every axis |
| wrong axis order | with $M\ne N$, wrong rectangle or OOB | fix `(p_m,p_n)` once and keep it |
| tail not masked | out-of-bounds read/write | mask both `tl.load` and `tl.store` |
| `BLOCK_N < N` without a loop | only the first columns are written | `BLOCK_N >= N` or a column loop / second axis |
| too many programs | idle or duplicate work | cap the grid at the tile count, or use grid-stride |
| grid-stride off-by-one | a tile is skipped or re-read | start `pB`, step `PB`, loop `< N` |
| mask with the wrong rank | `(B,) & (B,)` becomes `(B,B)` | `row_mask[:,None] & col_mask[None,:]` |
| multiple launches vs fusion | extra traffic and launch latency | fuse elementwise stages; keep launches for barriers |

## Self-check

1. For $A$ ($5\times7$) with $3\times4$ tiles, write the four `(pid_m,pid_n)` pairs, the rows and columns, and the masked cells.
2. Why is the grid-stride stride $PB$ rather than $P$ or $B$?
3. For $N=37$, $B=8$, $P=3$, list every start and the mask of the final tile.
4. Why does one program per column coalesce poorly on a row-major matrix?
5. A row kernel launches with `BLOCK_N=4` for a 7-column matrix. What is missing?
6. When is a two-launch pipeline better than one fused kernel?

## Summary

- The mapping is the design decision; the kernel body is mostly bookkeeping.
- The launch-grid rule is $P_k=\lceil \text{size}_k/B_k\rceil$ per axis.
- 1-D, 2-D, and 3-D grids are the same idea with more coordinates.
- Mask every non-divisible axis on both the load and the store.
- Grid-stride replaces a huge grid with a fixed number of persistent programs.
- Two launches buy a global barrier or a layout change; fusion buys traffic.

**Where this leads.** Chapter 19 turns the tile map into a full tiling strategy, and Chapter 20 fuses the multi-launch pipelines into one kernel.

## Exercises

1. **Column map.** Write the complete program-to-element table for $A$ ($5\times7$) with one program per column and $B_M=4$. Which lanes are masked?
2. **Tile sweep.** Recompute program count and mask waste for $B_M\times B_N\in\{1\times7,5\times1,5\times7,3\times4\}$. Which is best for $A$?
3. **3-D grid.** For $B=4$, $M=5$, $N=7$, $B_M=3$, $B_N=4$, write the grid and the offset of program $(3,1,1)$.
4. **Grid-stride.** Enumerate starts and masks for $N=35$, $B=4$, $P=3$. How many tiles and how many masked lanes?
5. **Boundary bug.** Delete the column mask from `tile_copy_kernel`, then run the $5\times7$ test. Which access fails first?
6. **Row block.** Change `row_scale` to `BLOCK_N=4` on $A$ and describe the wrong output. Fix it without changing `BLOCK_N`.
7. **Transpose offset.** Derive the store offset $jM+i$ from the load offset $iN+j$. What happens if you use $jN+i$?
8. **Launch cost.** For $M=N=1024$ and `float16`, compute the traffic of two launches versus one fused pass. At 10 µs per launch, what problem size makes the launch overhead dominate?
9. **Persistent grid.** Set $P$ to the number of resident programs (say 132 SMs) for $N=10^7$, $B=1024$. How many tiles per program, and what is the last mask?
10. **Attention map.** Scores have shape $(B,H,S,S)$. Choose a 3-D grid and explain which axis carries the batch and which carries the tile.

## Run everything

- **CPU.** The worked example and Method 01 run directly.
- **GPU.** Kernel tests are collected in one loop; without a GPU each one prints `SKIP <name>: ...` and the notebook continues.

In [ ]:
print("CPU tests")
test_method01()

print("\nGPU tests")
for test in (
    test_row_scale,
    test_column_scale,
    test_tile_copy,
    test_batch_copy,
    test_grid_stride,
    test_multiple_launches,
):
    run_if_gpu(test)